# Transcrito y proteína · Integrante 3

**Caso 1 · PAH | GRCh38 · NM_000277.3 · NP_000268.1**

Este notebook verifica las referencias, convierte coordenadas genómicas a posiciones
del CDS en la hebra negativa y calcula los cambios de codón y proteína de las 15
variantes del panel. Trata explícitamente la deleción VAR012, conserva la variante
no codificante y genera un FASTA por cada missense. Cada variante se analiza por
separado sobre WT; el panel no representa un haplotipo.

**Entradas:** VCF y GTF de `database/genomic/`, CDS de `database/transcript/`
y proteína WT de `database/protein/`. No se modifican los datos originales.

**Salidas:** `resultados/anotacion_transcrito_proteina.tsv`,
`resultados/mapa_cds.tsv` y `resultados/fasta/VAR00X_p.Cambio.fasta`.
No necesita la tabla de genómica, una API key ni modelos Evo2/RF3.

En Colab: **Entorno de ejecución → Ejecutar todo**. La configuración detecta
una copia local o descarga únicamente los datos públicos si hace falta.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import os

# Funciona desde la raíz, desde notebooks/ y en un Colab nuevo.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "Caso_1_PHA_Humano"]
ROOT = next((p for p in candidates if (p / "database/genomic/PAH_variants_masked_GRCh38.vcf").is_file()), None)
if ROOT is None:
    ROOT = cwd / "Caso_1_PHA_Humano"
    subprocess.run(["git", "clone", "https://github.com/XomiDessu/Caso_1_PHA_Humano.git", str(ROOT)], check=True)
os.chdir(ROOT)
missing = [package for module, package in [("Bio", "biopython"), ("pandas", "pandas")]
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import io
import re
import pandas as pd
import Bio
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils import seq3
from Bio.Align import substitution_matrices
from IPython.display import display

print(f"Referencias locales disponibles. Python {sys.version.split()[0]}; Biopython {Bio.__version__}; pandas {pd.__version__}")

Referencias locales disponibles. Python 3.13.9; Biopython 1.88; pandas 3.0.2


## 1. Verificar CDS y proteína WT

El CDS incluye el codón de terminación. `translate(cds=True)` comprueba inicio,
marco de lectura y paro terminal; su resultado debe coincidir con los 452 aa del FASTA WT.

In [2]:
cds_record = SeqIO.read("database/transcript/PAH_CDS_NM_000277.3.fasta", "fasta")
wt_record = SeqIO.read("database/protein/PAH_WT_NP_000268.1.fasta", "fasta")
cds = str(cds_record.seq).upper()
wt = str(wt_record.seq).upper()
assert cds_record.id.startswith("NM_000277.3|")
assert wt_record.id.startswith("NP_000268.1|")
assert len(cds) == 1359 and len(cds) % 3 == 0
assert cds.startswith("ATG") and cds[-3:] in {"TAA", "TAG", "TGA"}
assert len(wt) == 452
assert str(Seq(cds).translate(cds=True)) == wt
assert str(Seq(cds).translate()) == wt + "*"
display(pd.DataFrame([{"CDS_nt": len(cds), "proteina_aa": len(wt),
                       "inicio": cds[:3], "paro": cds[-3:], "traduccion_identica_WT": True}]))

   CDS_nt  proteina_aa inicio paro  traduccion_identica_WT
0    1359          452    ATG  TAA                    True


## 2. Mapa genoma → CDS en hebra negativa

VCF y GTF usan coordenadas desde 1 e intervalos cerrados. Se ordenan las 13
filas CDS por coordenada decreciente; dentro de cada segmento se recorre de
`end` a `start`, sin eliminar bases por la columna `frame`. La fase describe
la continuidad del codón y se verifica contra la longitud acumulada.
El mapa incluye el paro terminal del CDS proporcionado.

In [3]:
gtf_columns = ["seqname", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"]
gtf = pd.read_csv("database/genomic/PAH_NM_000277.3_GRCh38_subset.gtf",
                  sep="\t", comment="#", names=gtf_columns)
cds_segments = gtf.loc[(gtf.feature == "CDS") &
                        gtf.attribute.str.contains('transcript_id "NM_000277.3"', regex=False)].copy()
cds_segments = cds_segments.sort_values("start", ascending=False)
assert len(cds_segments) == 13
assert set(cds_segments.strand) == {"-"} and set(cds_segments.seqname) == {"chr12"}
g_to_c = {}
map_rows, segment_rows = [], []
for segment in cds_segments.itertuples():
    offset = len(g_to_c)
    assert int(segment.frame) == (-offset) % 3, "Fase CDS inconsistente"
    exon = int(re.search(r'exon_number "(\d+)"', segment.attribute).group(1))
    for genomic_pos in range(segment.end, segment.start - 1, -1):
        assert genomic_pos not in g_to_c
        c_pos = len(g_to_c) + 1
        g_to_c[genomic_pos] = c_pos
        map_rows.append({"CHROM": segment.seqname, "genomic_pos": genomic_pos,
                         "c_pos": c_pos, "exon": exon, "base_CDS": cds[c_pos - 1]})
    segment_rows.append({"exon": exon, "genomic_start": segment.start, "genomic_end": segment.end,
                         "c_start": offset + 1, "c_end": len(g_to_c), "fase": int(segment.frame)})
assert len(g_to_c) == len(cds) == 1359
assert list(g_to_c.values()) == list(range(1, 1360))
mapa_cds = pd.DataFrame(map_rows)
display(pd.DataFrame(segment_rows))

    exon  genomic_start  genomic_end  c_start  c_end  fase
0      1      102917071    102917130        1     60     0
1      2      102912791    102912898       61    168     0
2      3      102894735    102894918      169    352     0
3      4      102877462    102877550      353    441     2
4      5      102866596    102866663      442    509     0
5      6      102855136    102855332      510    706     1
6      7      102852815    102852950      707    842     2
7      8      102851687    102851756      843    912     1
8      9      102846895    102846951      913    969     0
9     10      102844336    102844431      970   1065     0
10    11      102843646    102843779     1066   1199     0
11    12      102840400    102840515     1200   1315     1
12    13      102839175    102839218     1316   1359     2


## 3. Alelos en orientación del transcrito

En SNV se usa el complemento; en alelos de varias bases se necesita el **reverso
complementario** y comprobar todo el intervalo REF. El alelo VCF de VAR012
incluye una base de anclaje que se conserva: diez bases REF frente a una ALT
significan nueve bases eliminadas, no diez.

Para variantes fuera del CDS no se inventa un codón ni un cambio proteico:
se conserva la fila con campos vacíos y una limitación explícita.

In [4]:
vcf_path = Path("database/genomic/PAH_variants_masked_GRCh38.vcf")
vcf = pd.read_csv(io.StringIO("".join(line for line in vcf_path.read_text().splitlines(keepends=True)
                                      if not line.startswith("##"))), sep="\t")
vcf = vcf.rename(columns={"#CHROM": "CHROM"})
assert len(vcf) == 15 and vcf.ID.is_unique
assert set(vcf.ID) == {f"VAR{i:03d}" for i in range(1, 16)}
assert set(vcf.CHROM) == {"chr12"}
assert all("," not in a for a in vcf.ALT)

oriented = []
for v in vcf.itertuples():
    genomic_positions = list(range(v.POS, v.POS + len(v.REF)))
    mapped = [g_to_c[g] for g in genomic_positions if g in g_to_c]
    if mapped:
        assert len(mapped) == len(v.REF), "Alelo cruza borde CDS; requiere tratamiento específico"
        positions = sorted(mapped)
        assert positions == list(range(positions[0], positions[-1] + 1))
        c_start, c_end = positions[0], positions[-1]
        ref_tx = str(Seq(v.REF).reverse_complement())
        alt_tx = str(Seq(v.ALT).reverse_complement())
        assert cds[c_start - 1:c_end] == ref_tx, f"REF no coincide con CDS: {v.ID}"
        oriented.append({"ID": v.ID, "c_start": c_start, "c_end": c_end,
                         "REF_transcrito": ref_tx, "ALT_transcrito": alt_tx, "REF_verificada": True})
    else:
        oriented.append({"ID": v.ID, "c_start": None, "c_end": None,
                         "REF_transcrito": None, "ALT_transcrito": None, "REF_verificada": None})
display(pd.DataFrame(oriented))

        ID  c_start   c_end REF_transcrito ALT_transcrito REF_verificada
0   VAR001   1278.0  1278.0              T              C           True
1   VAR002   1242.0  1242.0              C              T           True
2   VAR003   1222.0  1222.0              C              T           True
3   VAR004   1155.0  1155.0              C              G           True
4   VAR005   1013.0  1013.0              A              C           True
5   VAR006    963.0   963.0              C              T           True
6   VAR007    740.0   740.0              G              A           True
7   VAR008    735.0   735.0              G              A           True
8   VAR009    609.0   609.0              C              T           True
9   VAR010      NaN     NaN            NaN            NaN           None
10  VAR011    350.0   350.0              C              T           True
11  VAR012    310.0   319.0     GCCACTGTCC              C           True
12  VAR013    280.0   280.0              A         

## 4. Codón, aminoácido y consecuencia

Para SNV, codón = `(c − 1) // 3 + 1` y posición interna = `(c − 1) % 3`
(desde 0). Se verifica el residuo WT y se traduce también el CDS mutante completo.
En deleciones se retira el anclaje, se normaliza hacia 3′ y se compara la
proteína completa; no se trata la deleción como una SNV.

`protein_change` usa el formato corto del plan (`p.Arg408Trp`) para facilitar
la unión de tablas y los nombres de FASTA. `protein_change_predicho` añade
paréntesis HGVS (`p.(Arg408Trp)`) porque el efecto se deduce de ADN, sin validación
experimental de ARN/proteína. Las descripciones c. se refieren a NM_000277.3.

In [5]:
blosum62 = substitution_matrices.load("BLOSUM62")
def aa3(aa):
    return "Ter" if aa == "*" else seq3(aa)

def deletion_description(reference, alternate, prefix):
    # Mayor prefijo común = colocación más 3′ entre deleciones equivalentes.
    n_deleted = len(reference) - len(alternate)
    assert n_deleted > 0
    start0 = 0
    while start0 < len(alternate) and reference[start0] == alternate[start0]:
        start0 += 1
    end0 = start0 + n_deleted
    assert reference[:start0] + reference[end0:] == alternate, "No es deleción pura"
    if prefix == "c":
        location = str(start0 + 1) if n_deleted == 1 else f"{start0 + 1}_{end0}"
    else:
        first = f"{aa3(reference[start0])}{start0 + 1}"
        last = f"{aa3(reference[end0 - 1])}{end0}"
        location = first if n_deleted == 1 else f"{first}_{last}"
    return f"{prefix}.{location}del", start0, end0

annotations, mutant_proteins = [], {}
for v, tx in zip(vcf.itertuples(), oriented):
    assert v.ID == tx["ID"]
    row = {"ID": v.ID, "transcript_id": "NM_000277.3", "protein_id": "NP_000268.1",
           "CHROM": v.CHROM, "POS": v.POS, "REF": v.REF, "ALT": v.ALT, **tx,
           "transcript_change": None, "protein_change": None, "protein_change_predicho": None,
           "codon_number": None, "codon_position_0based": None, "codon_REF": None,
           "codon_ALT": None, "aa_WT": None, "aa_ALT": None,
           "consequence": None, "delta_nt": len(v.ALT) - len(v.REF), "delta_aa": None,
           "BLOSUM62": None, "limitaciones": "Predicción de secuencia; no demuestra efecto funcional ni patogenicidad."}
    if tx["c_start"] is None:
        row["consequence"] = "fuera_del_CDS"
        row["limitaciones"] = "Fuera del CDS: no se asigna cambio de codón/proteína; no se evalúa splicing ni expresión."
        annotations.append(row)
        continue

    start0, end0 = tx["c_start"] - 1, tx["c_end"]
    mutant_cds = cds[:start0] + tx["ALT_transcrito"] + cds[end0:]
    assert len(mutant_cds) - len(cds) == row["delta_nt"]
    # El panel contiene SNV y una deleción in-frame; otro tipo se detiene explícitamente.
    assert len(mutant_cds) % 3 == 0, "Frameshift no contemplado en este panel"
    mutant = str(Seq(mutant_cds).translate(cds=True))
    row["delta_aa"] = len(mutant) - len(wt)
    if len(v.REF) == len(v.ALT) == 1:
        c_pos = tx["c_start"]
        codon_index = (c_pos - 1) // 3
        within = (c_pos - 1) % 3
        codon_ref = cds[codon_index * 3:codon_index * 3 + 3]
        codon_alt = mutant_cds[codon_index * 3:codon_index * 3 + 3]
        aa_ref, aa_alt = str(Seq(codon_ref).translate()), str(Seq(codon_alt).translate())
        assert aa_ref == wt[codon_index]
        assert aa_alt == mutant[codon_index]
        row.update({"codon_number": codon_index + 1, "codon_position_0based": within,
                    "codon_REF": codon_ref, "codon_ALT": codon_alt,
                    "aa_WT": aa_ref, "aa_ALT": aa_alt,
                    "transcript_change": f"c.{c_pos}{tx['REF_transcrito']}>{tx['ALT_transcrito']}"})
        if aa_ref == aa_alt:
            row["consequence"] = "sinonima"
            row["protein_change"] = f"p.{aa3(aa_ref)}{codon_index + 1}="
            assert mutant == wt
        else:
            row["consequence"] = "missense"
            row["protein_change"] = f"p.{aa3(aa_ref)}{codon_index + 1}{aa3(aa_alt)}"
            row["BLOSUM62"] = int(blosum62[aa_ref, aa_alt])
            changed = [i + 1 for i, (a, b) in enumerate(zip(wt, mutant)) if a != b]
            assert len(mutant) == 452 and changed == [codon_index + 1]
            mutant_proteins[v.ID] = mutant
    else:
        assert v.ID == "VAR012" and len(v.REF) - len(v.ALT) == 9
        transcript_change, deleted_start, deleted_end = deletion_description(cds, mutant_cds, "c")
        protein_change, aa_start, aa_end = deletion_description(wt, mutant, "p")
        assert deleted_end - deleted_start == 9 and aa_end - aa_start == 3
        block_start = deleted_start // 3 * 3
        block_end = ((deleted_end + 2) // 3) * 3
        row.update({"transcript_change": transcript_change, "protein_change": protein_change,
                    "consequence": "delecion_in_frame", "codon_number": aa_start + 1,
                    "codon_position_0based": deleted_start % 3,
                    "codon_REF": cds[block_start:block_end], "codon_ALT": "",
                    "aa_WT": wt[aa_start:aa_end], "aa_ALT": "",
                    "limitaciones": "Deleción de 9 nt/3 aa, sin frameshift. codon_REF contiene el bloque eliminado; c_start/c_end incluyen el anclaje VCF. No se modela splicing."})
    row["protein_change_predicho"] = "p.(" + row["protein_change"][2:] + ")"
    annotations.append(row)

anotacion = pd.DataFrame(annotations)
for col in ["POS", "c_start", "c_end", "codon_number", "codon_position_0based", "delta_nt", "delta_aa", "BLOSUM62"]:
    anotacion[col] = anotacion[col].astype("Int64")
assert anotacion.ID.is_unique and len(anotacion) == 15
assert anotacion.consequence.value_counts().to_dict() == {
    "sinonima": 6, "missense": 7, "delecion_in_frame": 1, "fuera_del_CDS": 1}
display(anotacion[["ID", "transcript_change", "codon_REF", "codon_ALT", "protein_change_predicho", "consequence"]])
display(anotacion.consequence.value_counts().rename_axis("consequence").reset_index(name="n"))

        ID transcript_change  ... protein_change_predicho        consequence
0   VAR001         c.1278T>C  ...             p.(Asn426=)           sinonima
1   VAR002         c.1242C>T  ...             p.(Tyr414=)           sinonima
2   VAR003         c.1222C>T  ...           p.(Arg408Trp)           missense
3   VAR004         c.1155C>G  ...             p.(Leu385=)           sinonima
4   VAR005         c.1013A>C  ...           p.(Asp338Ala)           missense
5   VAR006          c.963C>T  ...             p.(Leu321=)           sinonima
6   VAR007          c.740G>A  ...           p.(Gly247Asp)           missense
7   VAR008          c.735G>A  ...             p.(Val245=)           sinonima
8   VAR009          c.609C>T  ...             p.(Cys203=)           sinonima
9   VAR010               NaN  ...                     NaN      fuera_del_CDS
10  VAR011          c.350C>T  ...           p.(Thr117Ile)           missense
11  VAR012      c.310_318del  ...    p.(Ala104_Val106del)  delecion_in_frame

## 5. FASTA de los siete missense

Se sustituye exactamente un residuo sobre WT y se vuelve a leer cada archivo
para verificar longitud, identidad y posición. Se conserva el nombre del plan
sin paréntesis. La deleción no se exporta como missense; su consecuencia se
documenta en la tabla para que el equipo decida su eventual modelado por separado.

In [6]:
fasta_dir = Path("resultados/fasta")
fasta_dir.mkdir(parents=True, exist_ok=True)
fasta_checks = []
for row in anotacion.loc[anotacion.consequence == "missense"].itertuples():
    position = int(row.codon_number)
    mutant = wt[:position - 1] + row.aa_ALT + wt[position:]
    assert mutant == mutant_proteins[row.ID]
    differences = [i + 1 for i, (a, b) in enumerate(zip(wt, mutant)) if a != b]
    assert len(mutant) == 452 and differences == [position]
    name = f"{row.ID}_{row.protein_change}"
    fasta_path = fasta_dir / f"{name}.fasta"
    record = SeqRecord(Seq(mutant), id=name,
                       description="PAH|NP_000268.1|prediccion_in_silico|452_aa")
    SeqIO.write(record, str(fasta_path), "fasta")
    reread = SeqIO.read(str(fasta_path), "fasta")
    assert str(reread.seq) == mutant
    fasta_checks.append({"ID": row.ID, "archivo": fasta_path.as_posix(),
                         "longitud_aa": len(mutant), "posicion_cambiada": differences[0]})
assert len(fasta_checks) == 7
display(pd.DataFrame(fasta_checks))

       ID  ... posicion_cambiada
0  VAR003  ...               408
1  VAR005  ...               338
2  VAR007  ...               247
3  VAR011  ...               117
4  VAR013  ...                94
5  VAR014  ...                76
6  VAR015  ...                24

[7 rows x 4 columns]


## 6. Guardar resultados y verificar ejemplos manuales

Tres controles por cálculo directo, independientes del diccionario de coordenadas:

- **VAR003:** antes del exón 12 hay 1,199 nt; `1199 + (102840515 − 102840493 + 1) = 1222`.
  G>A genómico → C>T; CGG→TGG; Arg408→Trp.
- **VAR007:** antes del exón 7 hay 706 nt; `706 + (102852950 − 102852917 + 1) = 740`.
  C>T → G>A; GGC→GAC; Gly247→Asp.
- **VAR012:** antes del exón 3 hay 168 nt. REF genómico abarca
  102894768–102894777 → c.310–319; reverso complementario GCCACTGTCC→C.
  Se conserva la C final y se elimina c.310_318 (GCC ACT GTC → Ala104, Thr105, Val106).

In [7]:
by_id = anotacion.set_index("ID")
assert 1199 + (102840515 - 102840493 + 1) == 1222
assert (by_id.loc["VAR003", ["transcript_change", "codon_REF", "codon_ALT", "protein_change"]].tolist()
        == ["c.1222C>T", "CGG", "TGG", "p.Arg408Trp"])
assert 706 + (102852950 - 102852917 + 1) == 740
assert (by_id.loc["VAR007", ["transcript_change", "codon_REF", "codon_ALT", "protein_change"]].tolist()
        == ["c.740G>A", "GGC", "GAC", "p.Gly247Asp"])
assert 168 + (102894918 - 102894777 + 1) == 310
assert (by_id.loc["VAR012", ["transcript_change", "protein_change", "delta_nt", "delta_aa"]].tolist()
        == ["c.310_318del", "p.Ala104_Val106del", -9, -3])

output_dir = Path("resultados")
output_dir.mkdir(exist_ok=True)
anotacion.to_csv(output_dir / "anotacion_transcrito_proteina.tsv", sep="\t", index=False, na_rep="")
mapa_cds.to_csv(output_dir / "mapa_cds.tsv", sep="\t", index=False)
reloaded = pd.read_csv(output_dir / "anotacion_transcrito_proteina.tsv", sep="\t")
assert len(reloaded) == 15 and reloaded.ID.is_unique
assert reloaded.loc[reloaded.ID == "VAR010", "protein_change"].isna().all()
print("VALIDADO: 1,359 nt; WT 452 aa; 15 variantes; 7 FASTA missense; 3 controles manuales.")
print("Exportados: resultados/anotacion_transcrito_proteina.tsv y resultados/mapa_cds.tsv")

VALIDADO: 1,359 nt; WT 452 aa; 15 variantes; 7 FASTA missense; 3 controles manuales.
Exportados: resultados/anotacion_transcrito_proteina.tsv y resultados/mapa_cds.tsv


## Cierre: observación, inferencia y límites

- **Observamos:** el CDS de 1,359 nt traduce exactamente a la proteína WT de 452 aa.
  Las 14 variantes con REF en CDS coinciden con la referencia tras orientar los alelos.
- **Inferimos que:** seis variantes son sinónimas, siete son missense y una es una
  deleción in-frame de nueve nt/tres aa. VAR010 queda fuera del CDS.
- **La evidencia aporta:** posiciones c., codones, cambios proteicos y siete secuencias
  mutantes reproducibles para el trabajo posterior.
- **Concluimos:** el mapeo y los efectos sobre la secuencia están comprobados para
  este transcrito; no determinan por sí solos el efecto funcional o clínico.
- **Falta evidencia:** ensayos de ARN/splicing, expresión y actividad enzimática,
  además de las capas Evo2 y estructural del equipo. Una variante sinónima o fuera
  del CDS no se considera inocua por esta anotación.

### Referencias metodológicas

Datos locales: `PAH_variants_masked_GRCh38.vcf`, `PAH_NM_000277.3_GRCh38_subset.gtf`,
`PAH_CDS_NM_000277.3.fasta`, `PAH_WT_NP_000268.1.fasta`.

- [Biopython: secuencias, reverso complementario y traducción](https://biopython.org/wiki/Seq).
- [HGVS: lista de comprobación y consecuencias predichas](https://hgvs-nomenclature.org/stable/recommendations/checklist/).
- [HGVS: deleciones en ADN y regla 3′](https://hgvs-nomenclature.org/stable/recommendations/DNA/deletion/).
- [HGVS: deleciones en proteína](https://hgvs-nomenclature.org/stable/recommendations/protein/deletion/).

### Revisiones del Integrante 3

Los cálculos manuales anteriores sirven para la revisión cruzada. La revisión
del notebook del Integrante 2 y del ranking requiere que esos notebooks estén
completos. La conciliación final de POPAF, ΔlogP, RMSD y rankings con reporte,
presentación y notebook integrado queda pendiente de los entregables del equipo.